In [ ]:
import sys
from pathlib import Path

ML_ROOT = Path.cwd().parent

if str(ML_ROOT) not in sys.path:
    sys.path.append(str(ML_ROOT))

print(ML_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import wfdb
from collections import Counter

from src.datasets.heartbeat_extractor import extract_heartbeat
from src.datasets.record_processor import process_record

In [ ]:
record = wfdb.rdrecord("../data/raw/mitdb/100")
annotation = wfdb.rdann("../data/raw/mitdb/100", "atr")

signal = record.p_signal[:, 0]

In [ ]:
from src.datasets.dataset_builder import build_dataset
from src.config.constants import MITBIH_RECORDS

X, y, patient_ids = build_dataset(MITBIH_RECORDS)

print(X.shape)
print(y.shape)
print(patient_ids.shape)
print(np.unique(patient_ids))

In [ ]:
unique_labels, counts = np.unique(y, return_counts=True)

for label, count in zip(unique_labels, counts):
    print(label, count)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "patient_id": patient_ids,
    "label": y
})

patient_distribution = (
    df.groupby(["patient_id", "label"])
      .size()
      .unstack(fill_value=0)
)

patient_distribution

In [ ]:
import wfdb
import pandas as pd

record_path = "../data/raw/mitdb/102"

annotation = wfdb.rdann(record_path, "atr")

pd.Series(annotation.symbol).value_counts()

In [ ]:
from src.preprocessing.splitter import get_patient_splits

train_patients, val_patients, test_patients = get_patient_splits(
    y,
    patient_ids,
    regenerate=True,
)

print("Train:", train_patients)
print("Validation:", val_patients)
print("Test:", test_patients)

print()

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

In [ ]:
train_mask = np.isin(patient_ids, train_patients)
val_mask = np.isin(patient_ids, val_patients)
test_mask = np.isin(patient_ids, test_patients)

print("Train")
print(pd.Series(y[train_mask]).value_counts())
print()

print("Validation")
print(pd.Series(y[val_mask]).value_counts())
print()

print("Test")
print(pd.Series(y[test_mask]).value_counts())